In [ ]:
import numpy as np
import itertools
from typing import Optional
from warnings import warn
from copy import deepcopy

In [ ]:
class Stratification:
    def __init__(self, name: str, strata: list[str]):
        self.name = name
        self.strata = strata

    def __repr__(self):
        return f"Stratification: {self.name}"

In [ ]:
StratSpec = tuple[Stratification, str] | tuple[Stratification, list[str]] | None

In [ ]:
from proto import Compartment

In [ ]:
class Compartment:
    def __init__(self, strata: list[tuple[Stratification, str]], index: int):
        self.strata = strata
        self.index = index

    def __repr__(self):
        return "Compartment :" + repr(self.strata)


class CompartmentMap:
    def __init__(self, compartments, stratifications):
        self.compartments: list[Compartment] = compartments
        self.stratifications: dict[
            Stratification : tuple[Stratification, list[str]]
        ] = stratifications
        self._base_strat = list(stratifications)[0]

    @classmethod
    def new(cls, base_strat: Stratification):
        compartments = [
            Compartment([(base_strat, s)], i) for i, s in enumerate(base_strat.strata)
        ]
        stratifications: dict[Stratification, Optional[tuple]] = {base_strat: None}
        return cls(compartments, stratifications)

    def stratify(self, strat: Stratification, stratifies: StratSpec = None):

        if stratifies is None:
            stratifies = (self._base_strat, self._base_strat.strata)
        if isinstance(stratifies[1], str):
            stratifies = (stratifies[0], [stratifies[1]])

        target_strat, target_strata = stratifies

        for existing_strat, estrat_stratifies in self.stratifications.items():
            if existing_strat.name == strat.name:
                warn(f"Existing stratification with name {strat.name}")
                # +++ Actually check for overlap, not just equivalency
                if estrat_stratifies == stratifies:
                    raise Exception(
                        "Existing stratification with same name overlaps",
                        strat.name,
                        stratifies,
                    )
        out_comps = []
        new_comps = []
        i = 0

        for c in self.compartments:
            if any(
                [(target_strat, t_stratum) in c.strata for t_stratum in target_strata]
            ):
                for stratum in strat.strata:
                    new_c = Compartment(c.strata + [(strat, stratum)], i)
                    out_comps.append(new_c)
                    new_comps.append(new_c)
                    i += 1
            else:
                out_comps.append(c)
                i += 1

        if len(new_comps) == 0:
            raise Exception("No compartments match stratification request", stratifies)

        self.compartments = out_comps
        self.stratifications[strat] = stratifies
        return strat

    def rebase(
        self, new_base_strat: Stratification, key, in_place=False
    ) -> "CompartmentMap":
        new_stratifications = {new_base_strat: None}

        for k, v in self.stratifications.items():
            if k is self._base_strat:
                new_stratifications[k] = (k, [key])
            else:
                new_stratifications[k] = v
        # self.stratifications = new_stratifications
        new_c = [
            Compartment([(new_base_strat, key)] + c.strata, c.index)
            for c in self.compartments
        ]

        if in_place:
            self.stratifications = new_stratifications
            self._base_strat = new_base_strat
            self.compartments = new_c
            return self
        else:
            return CompartmentMap(new_c, new_stratifications)

    def add_compartments(self, other_comps: "CompartmentMap", base_stratum: str):
        idx = len(self.compartments)
        assert not any(
            [other_s in self.stratifications for other_s in other_comps.stratifications]
        )

        if base_stratum not in self._base_strat.strata:
            raise KeyError("Stratum not found in base stratification", base_stratum)

        other_rebased = other_comps.rebase(self._base_strat, base_stratum)

        for c in other_comps.compartments:
            new_c = Compartment([(self._base_strat, base_stratum)] + c.strata, idx)
            self.compartments.append(new_c)
            idx += 1

        for k, v in other_rebased.stratifications.items():
            self.stratifications[k] = v

In [ ]:
pop_strat = Stratification("pop", ["wolf", "human"])
cm = CompartmentMap.new(pop_strat)

In [ ]:
wolf_class_strat = cm.stratify(
    Stratification("wolf_class", ["A", "B"]), (pop_strat, "wolf")
)
disease_state_strat = cm.stratify(
    Stratification("disease_state", ["S", "I", "R"]), (pop_strat, "human")
)
h_age_strat = cm.stratify(
    Stratification("age", ["child", "young_adult", "adult", "older"]),
    (pop_strat, "human"),
)

In [ ]:
work_risk_strat = cm.stratify(
    Stratification("work_risk", ["low", "high"]),
    (h_age_strat, ["young_adult", "adult"]),
)
w_age_strat = cm.stratify(
    Stratification("age", ["juvenile", "adult"]), (pop_strat, "wolf")
)
severity_strat = cm.stratify(
    Stratification("severity", ["asymp", "mild", "severe"]), (disease_state_strat, "I")
)

In [ ]:
cm.stratifications

In [ ]:
cm.compartments

In [ ]:
disease_state_strat = Stratification("disease_state", ["S", "I", "R"])
humans = CompartmentMap.new(disease_state_strat)
humans.compartments

In [ ]:
h_age_strat = humans.stratify(
    Stratification("age", ["child", "young_adult", "adult", "older"])
)
work_risk_strat = humans.stratify(
    Stratification("work_risk", ["low", "high"]),
    (h_age_strat, ["young_adult", "adult"]),
)
severity_strat = humans.stratify(
    Stratification("severity", ["asymp", "mild", "severe"]), (disease_state_strat, "I")
)

humans.compartments

In [ ]:
wolf_class_strat = Stratification("wolf_class", ["A", "B"])
wolves = CompartmentMap.new(wolf_class_strat)
w_age_strat = wolves.stratify(Stratification("age", ["juvenile", "adult"]))
wolves.compartments

In [ ]:
humans.compartments

In [ ]:
pop_strat = Stratification("pop", ["human", "wolf"])
rebased = humans.rebase(pop_strat, "human")

In [ ]:
rebased.compartments

In [ ]:
rebased.add_compartments(wolves, "wolf")

In [ ]:
rebased.compartments

In [ ]:
def cmap_to_ctable(compartments):
    strat_compartment_indices = {}
    strat_comp_strata = {}

    for i, c in enumerate(compartments):
        for strat, stratum in c.strata:
            # print(c,strat,stratum)
            strat_comp_idx_l = strat_compartment_indices.setdefault(strat, [])
            strat_comp_strata_l = strat_comp_strata.setdefault(strat, [])
            strat_comp_idx_l.append(i)
            strat_comp_strata_l.append(strat.strata.index(stratum))

    strat_compartment_indices = {
        k: np.array(v, dtype=int) for k, v in strat_compartment_indices.items()
    }
    strat_comp_strata = {
        k: np.array(v, dtype=int) for k, v in strat_comp_strata.items()
    }

    return strat_compartment_indices, strat_comp_strata

In [ ]:
strat_comp_indices, strat_comp_strata = cmap_to_ctable(rebased.compartments)

In [ ]:
strat_comp_indices, strat_comp_strata

In [ ]:
comps[[0, 0, 0, 1, 1, 1]]

In [ ]:
cmap_to_ctable(comps[[0, 0, 0, 1, 1, 1]])

In [ ]:
strat_comp_strata

In [ ]:
np.isin(strat_comp_strata[disease_state_strat], [2, 3])

In [ ]:
strat_comp_indices[disease_state_strat][
    strat_comp_strata[disease_state_strat] in [2, 3]
]

In [ ]:
comps = np.array(rebased.compartments)

In [ ]:
comps[wolf_class_strat]

In [ ]:
# Flow broadcasting
# Should this be a property of flows, or stratifications, or something else?

# infection: S->I
#
# S->[I_asymp, I_mild, I_severe]
# Direct adjustment (split by "stratification population")
# Multi_adjustment; older populations end up in severe more often
# Do we disregard strat population altogether? Have an agexseverity transition table?
# If we stratify first, then adjustment makes more sense "with the flow"
# If we set the flow first, this doesn't make sense

In [ ]:
list(itertools.permutations((a, b, c, d)))

In [ ]:
a = np.arange(16)
b = np.arange(3)
c = np.arange(9)
d = np.arange(256)

len(list(itertools.product(a, b, c, d)))

# 6groups, each 2mul; 12mul
# 8groups, each 3mul, 19mul

In [ ]:
infection: agegroup

In [ ]:
import jax
from jax import numpy as jnp

In [ ]:
a.reshape((a.shape[0], 1))

In [ ]:
ar = a.reshape((a.shape[0], 1)) * 1.1
dr = d.reshape((1, d.shape[0])) * 0.1

jax.make_jaxpr(jnp.multiply)(ar, dr)

In [ ]:
np.unique(list(zip(a, b)), axis=0)

In [ ]:
class TransitionFlow:
    def __init__(self, name, src_query, dst_query):
        

In [ ]:
# Stratifications:

In [ ]:
def cmap_to_ctable(cmap: CompartmentMap):
    strat_compartment_indices = {}
    strat_comp_strata = {}

    for i, c in enumerate(cmap.compartments):
        for strat, stratum in c.strata:
            # print(c,strat,stratum)
            strat_comp_idx_l = strat_compartment_indices.setdefault(strat, [])
            strat_comp_strata_l = strat_comp_strata.setdefault(strat, [])
            strat_comp_idx_l.append(i)
            strat_comp_strata_l.append(strat.strata.index(stratum))

    strat_compartment_indices = {
        k: np.array(v, dtype=int) for k, v in strat_compartment_indices.items()
    }
    strat_comp_strata = {
        k: np.array(v, dtype=int) for k, v in strat_comp_strata.items()
    }

    return strat_compartment_indices, strat_comp_strata

In [ ]:
sci, scs = cmap_to_ctable(sm)
sci, scs

In [ ]:
sm.compartments[sci[wolf_class_strat][scs[wolf_class_strat] == 1]]

In [ ]:
sm.compartments

In [ ]:
# something like numpy axes, but not quite...
# mapping tables
# "age_compartments": [indices of all compartments stratified by age]
# "age_comp_strata": [values of which age stratum said compartments contain]

In [ ]:
class CompartmentData:
    def __init__(self, cmap, data):
        self.cmap = cmap
        self.data = data